# RealSaS — TESSA Knight T1 Run-All

FIT1 supervised topology witness for **TESSA = Topology Estimation for Stable Surface Animation**.

This notebook mounts Drive, verifies the exact GSA + raw-41 teacher-mesh handoff, clones the frozen research branch, runs a reduced-model scientific preflight, then fits the full TESSA model on one **global teacher topology sequence** using overlapping causal training windows. Artificial chart splitting is forbidden in the canonical T1 path.

It does **not** claim product authority or generalization. T1 is supervised topology fit only; T2 mechanical-consequence training and hard Compiler courts remain mandatory.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

BRANCH='research/tessa-mechanical-mesh-v1-20261004'
REPO_URL='https://github.com/merynz/RealSaS-OPT.git'
HANDOFF=Path('/content/drive/MyDrive/RealSaS_TESSA_KNIGHT_V1_HANDOFF')
DRIVE_OUTPUT=Path('/content/drive/MyDrive/RealSaS_TESSA_KNIGHT_V1_OUTPUT')
WORK=Path('/content/realsas_tessa_knight_v1')
SRC=WORK/'source'
LOCAL_OUTPUT=WORK/'output'

def sha256(path: Path) -> str:
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

manifest_path=HANDOFF/'TESSA_HANDOFF_MANIFEST.json'
assert manifest_path.is_file(), f'MISSING_HANDOFF_MANIFEST:{manifest_path}'
manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['schema']=='RealSaS.TESSAKnightTrainingHandoff.v1', manifest
assert manifest['status']=='PASS', manifest
assert manifest['product_authority_claimed'] is False, manifest
assert manifest['teacher_contract']['bones']==41, manifest
assert manifest['teacher_contract']['zero_skin_bone_deletion_forbidden'] is True, manifest
for rel,meta in manifest['files'].items():
    p=HANDOFF/rel
    assert p.is_file(), f'HANDOFF_FILE_MISSING:{rel}'
    assert p.stat().st_size==int(meta['bytes']), f'HANDOFF_SIZE_DRIFT:{rel}'
    assert sha256(p)==meta['sha256'], f'HANDOFF_SHA_DRIFT:{rel}'
SURFACE_JSON=HANDOFF/manifest['roles']['rigging_surface']
TEACHER_NPZ=HANDOFF/manifest['roles']['teacher_source_truth']
print('TESSA_HANDOFF_VERIFIED', manifest.get('source_repo_commit'), sha256(manifest_path))


In [ ]:
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
LOCAL_OUTPUT.mkdir(parents=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(SRC)],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-input','--progress-bar','off','-r',str(SRC/'requirements/mainline-ci.txt')],check=True)
import torch
assert torch.cuda.is_available(), 'TESSA_A100_GPU_REQUIRED'
print('TESSA_GPU_PREFLIGHT', torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))


In [ ]:
env=dict(os.environ)
env['PYTHONPATH']=str(SRC)+((':'+env['PYTHONPATH']) if env.get('PYTHONPATH') else '')
smoke_dir=WORK/'smoke'
cmd=[sys.executable,str(SRC/'tools/training/run_tessa_supervised_fit_v1.py'),'--surface-json',str(SURFACE_JSON),'--teacher-npz',str(TEACHER_NPZ),'--output-dir',str(smoke_dir),'--epochs','1','--target-tokens','256','--grad-accum','1','--small-smoke']
subprocess.run(cmd,cwd=SRC,env=env,check=True)
smoke=json.loads((smoke_dir/'TESSA_V1_FIT_RESULT.json').read_text(encoding='utf-8'))
assert smoke['status']=='PASS', smoke
m=smoke['input_manifest']
assert m['artificial_topology_chart_split'] is False, m
print('TESSA_SMALL_SMOKE_PASS', {'sequence_tokens':m['global_sequence_tokens'],'windows':m['training_window_count'],'components':m['teacher_component_count']})


In [ ]:
LOCAL_OUTPUT.mkdir(parents=True,exist_ok=True)
if DRIVE_OUTPUT.exists() and (DRIVE_OUTPUT/'TESSA_V1_LATEST.pt').is_file():
    shutil.copytree(DRIVE_OUTPUT,LOCAL_OUTPUT,dirs_exist_ok=True)
    print('TESSA_RESUME_PAYLOAD_COPIED',DRIVE_OUTPUT)

cmd=[sys.executable,str(SRC/'tools/training/run_tessa_supervised_fit_v1.py'),'--surface-json',str(SURFACE_JSON),'--teacher-npz',str(TEACHER_NPZ),'--output-dir',str(LOCAL_OUTPUT),'--epochs','20','--lr','2e-4','--weight-decay','0.01','--target-tokens','4096','--grad-accum','4','--seed','1337']
if (LOCAL_OUTPUT/'TESSA_V1_LATEST.pt').is_file(): cmd.append('--resume')
subprocess.run(cmd,cwd=SRC,env=env,check=True)


In [ ]:
result_path=LOCAL_OUTPUT/'TESSA_V1_FIT_RESULT.json'
assert result_path.is_file(), 'TESSA_RESULT_MISSING'
result=json.loads(result_path.read_text(encoding='utf-8'))
assert result['schema']=='RealSaS.TESSASupervisedFitResult.v1', result
assert result['status']=='PASS', result
assert result['training_class']=='T1_SUPERVISED_TOPOLOGY_FIT', result
assert result['input_manifest']['artificial_topology_chart_split'] is False, result
assert result['product_authority_claimed'] is False, result
assert result['generalization_claimed'] is False, result
assert result['mechanical_consequence_training_included'] is False, result
assert result['next_required_rung']=='T2_MECHANICAL_CONSEQUENCE_FIT', result
if DRIVE_OUTPUT.exists(): shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(LOCAL_OUTPUT,DRIVE_OUTPUT)
marker={'schema':'RealSaS.TESSAKnightT1RunAllComplete.v1','status':'PASS','handoff_manifest_sha256':sha256(manifest_path),'fit_result_sha256':sha256(DRIVE_OUTPUT/'TESSA_V1_FIT_RESULT.json'),'checkpoint_sha256':result['latest_checkpoint_sha256'],'global_sequence_tokens':result['input_manifest']['global_sequence_tokens'],'training_window_count':result['input_manifest']['training_window_count'],'artificial_topology_chart_split':False,'product_authority_claimed':False,'generalization_claimed':False,'next_required_rung':'T2_MECHANICAL_CONSEQUENCE_FIT'}
(DRIVE_OUTPUT/'RUN_ALL_COMPLETE.json').write_text(json.dumps(marker,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print('TESSA_KNIGHT_T1_RUN_ALL_PASS')
print(json.dumps(marker,sort_keys=True))
